# Демонстрация логики анализа ассортиментной матрицы бьюти-товаров 💄

Привет! Рада видеть вас в моем репозитории. В этом ноутбуке я покажу **упрощенную логику расчетов**, которую я использовала в реальном коммерческом проекте.

Поскольку реальные данные защищены NDA (соглашением о неразглашении), для этой демонстрации мы сгенерируем синтетический датасет бьюти-товаров с помощью Python, а затем рассчитаем ключевые операционные метрики ритейла:
* Динамику выручки месяц к месяцу (MoM)
* Процент выкупа товаров клиентами
* Топ-лидеров и топ-аутсайдеров ассортиментной матрицы

### Шаг 1. Импорт библиотек и генерация тестовых данных
Создадим таблицу продаж косметики за два месяца (январь и февраль 2026 года).

In [ ]:
import pandas as pd
import numpy as np

# Фиксируем генератор случайных чисел для воспроизводимости результатов
np.random.seed(42)

# Сгенерируем список бьюти-товаров
brands = ['Loreal', 'Estee Lauder', 'Shiseido', 'Chanel', 'Dior', 'MAC', 'Clinique']
categories = ['Уход за кожей', 'Декоративная косметика', 'Парфюмерия', 'Уход за волосами']

n_rows = 300
raw_data = {
    'order_id': np.random.randint(10000, 99999, n_rows),
    'month': np.random.choice(['Январь', 'Февраль'], n_rows, p=[0.45, 0.55]),
    'brand': np.random.choice(brands, n_rows),
    'category': np.random.choice(categories, n_rows),
    'price': np.random.randint(500, 8000, n_rows),
    'quantity': np.random.randint(1, 4, n_rows),
    # Выкуп товара (был ли заказ оплачен в ПВЗ/курьеру). Поставим базовую вероятность выкупа 92%
    'is_bought': np.random.choice([1, 0], n_rows, p=[0.92, 0.08])
}

df = pd.DataFrame(raw_data)
# Считаем общую стоимость заказа до выкупа
df['total_order_value'] = df['price'] * df['quantity']

print("--- Первые 5 строк нашего синтетического датасета ---")
df.head()

### Шаг 2. Расчет ключевых KPI: Выручка и Выкуп
**Выручка** — это стоимость только тех товаров, которые клиент реально выкупил (`is_bought == 1`).
**Процент выкупа** — это отношение реальной выручки к общей стоимости всех заказанных товаров (целевой ориентир бизнеса — 95%).

In [ ]:
# Добавим колонку реальной выручки
df['revenue'] = df['total_order_value'] * df['is_bought']

# Группируем данные по месяцам для оценки динамики
monthly_kpi = df.groupby('month').agg(
    total_ordered=('total_order_value', 'sum'),
    actual_revenue=('revenue', 'sum'),
    items_sold=('quantity', 'sum')
).reindex(['Январь', 'Февраль'])

# Считаем процент выкупа
monthly_kpi['buyout_rate_%'] = round((monthly_kpi['actual_revenue'] / monthly_kpi['total_ordered']) * 100, 2)

print("--- Операционные показатели по месяцам ---")
monthly_kpi

### Шаг 3. Анализ динамики месяц к месяцу (MoM)
Посмотрим, как изменилась выручка в Феврале по сравнению с Январем.

In [ ]:
jan_rev = monthly_kpi.loc['Январь', 'actual_revenue']
feb_rev = monthly_kpi.loc['Февраль', 'actual_revenue']

mom_growth = round(((feb_rev - jan_rev) / jan_rev) * 100, 2)
print(f"Прирост выручки в Феврале по сравнению с Январем (MoM): {mom_growth}%")

### Шаг 4. Факторный анализ: Поиск лидеров и аутсайдеров
Для бизнеса критически важно знать, какие бренды приносят максимум денег, а какие тянут экономику вниз из-за низкого выкупа (например, дорогие духи, которые часто заказывают, но отказываются при получении).

In [ ]:
# Агрегируем данные по брендам
brand_analytics = df.groupby('brand').agg(
    ordered_value=('total_order_value', 'sum'),
    revenue=('revenue', 'sum'),
    orders_count=('order_id', 'count')
)

brand_analytics['buyout_rate_%'] = round((brand_analytics['revenue'] / brand_analytics['ordered_value']) * 100, 2)

# Топ-3 лидера по выручке
leaders = brand_analytics.sort_values(by='revenue', ascending=False).head(3)

# Топ-3 аутсайдера по проценту выкупа (проблемные контракты)
outsiders = brand_analytics.sort_values(by='buyout_rate_%', ascending=True).head(3)

print("🌟 ТОП-3 БРЕНДА ПО ВЫРУЧКЕ:")
print(leaders[['revenue', 'buyout_rate_%']])

print("
🚨 ТОП-3 АУТСАЙДЕРА С НИЗКИМ ВЫКУПОМ (Зона риска):")
print(outsiders[['revenue', 'buyout_rate_%']])

### Выводы для руководства (CEO):
1. **Динамика продаж:** Целевой показатель роста выручки (план +5%) перевыполнен.
2. **Проблема выкупа:** Бренды в зоне риска имеют процент выкупа ниже целевых 95%. Для них рекомендуется ввести частичную предоплату на сайте или запустить точечные скидки, чтобы снизить процент отказов в ПВЗ.